# 01 — Pricing European Options

This notebook shows how to price a **European vanilla option** using every
pricing method available in `derivatives_pricing`:

1. **Black-Scholes-Merton** (analytical)
2. **Binomial tree** (CRR)
3. **PDE finite difference** (Crank-Nicolson)
4. **Monte Carlo** simulation

We also demonstrate pricing with a **continuous dividend yield** and
**discrete cash dividends**.

## 1) Setup

All pricing flows follow the same pattern:

```
DiscountCurve → MarketData → UnderlyingData → OptionValuation.present_value()
```

In [1]:
import datetime as dt

import derivatives_pricing as dp

## 2) Market Data

We set up a flat risk-free rate of 5 % and a pricing date of 15 Jun 2025.

In [2]:
pricing_date = dt.datetime(2025, 6, 15)
r = 0.05
maturity = dt.datetime(2026, 6, 15)  # 1-year option

curve = dp.DiscountCurve.flat(rate=r, end_time=2.0)
market = dp.MarketData(pricing_date=pricing_date, discount_curve=curve, currency="USD")

## 3) Underlying & Contract

A European call on a stock trading at 100 with 20 % vol, struck at 105.

In [3]:
underlying = dp.UnderlyingData(
    initial_value=100.0,
    volatility=0.20,
    market_data=market,
)

spec = dp.VanillaSpec(
    option_type=dp.OptionType.CALL,
    exercise_type=dp.ExerciseType.EUROPEAN,
    strike=105.0,
    maturity=maturity,
)

## 4) Black-Scholes-Merton (Analytical)

In [4]:
bsm = dp.OptionValuation(underlying=underlying, spec=spec, pricing_method=dp.PricingMethod.BSM)
print(f"BSM price:      {bsm.present_value():.4f}")

BSM price:      8.0214


## 5) Binomial Tree

In [5]:
binom = dp.OptionValuation(
    underlying=underlying,
    spec=spec,
    pricing_method=dp.PricingMethod.BINOMIAL,
    params=dp.BinomialParams(num_steps=500),
)
print(f"Binomial price: {binom.present_value():.4f}")

Binomial price: 8.0232


## 6) PDE Finite Difference (Crank-Nicolson)

In [6]:
pde = dp.OptionValuation(
    underlying=underlying,
    spec=spec,
    pricing_method=dp.PricingMethod.PDE_FD,
    params=dp.PDEParams(),  # defaults: Crank-Nicolson + Rannacher smoothing
)
print(f"PDE price:      {pde.present_value():.4f}")

PDE price:      8.0213


## 7) Monte Carlo

For MC we need a `GBMProcess` (path simulator) instead of `UnderlyingData`.

In [7]:
sim_config = dp.SimulationConfig(paths=100_000, end_date=maturity, num_steps=100)
gbm = dp.GBMProcess(
    market_data=market,
    process_params=dp.GBMParams(initial_value=100.0, volatility=0.20),
    sim_config=sim_config,
)

mc = dp.OptionValuation(
    underlying=gbm,
    spec=spec,
    pricing_method=dp.PricingMethod.MONTE_CARLO,
    params=dp.MonteCarloParams(random_seed=42),
)
print(f"MC price:       {mc.present_value():.4f}")

MC price:       8.0893


## 8) Side-by-Side Comparison

All four methods should converge to the same analytical BSM value.

In [8]:
bsm_pv = bsm.present_value()
print(f"{'Method':<20} {'Price':>10} {'Diff vs BSM':>12}")
print("-" * 44)
for label, val in [
    ("BSM", bsm),
    ("Binomial", binom),
    ("PDE (C-N)", pde),
    ("Monte Carlo", mc),
]:
    pv = val.present_value()
    print(f"{label:<20} {pv:>10.4f} {pv - bsm_pv:>+12.4f}")

Method                    Price  Diff vs BSM
--------------------------------------------
BSM                      8.0214      +0.0000
Binomial                 8.0232      +0.0018
PDE (C-N)                8.0213      -0.0000
Monte Carlo              8.0893      +0.0680


## 9) Continuous Dividend Yield

A continuous dividend yield is modelled via a `dividend_curve` on the
underlying. This enters the drift of the risk-neutral process.

In [9]:
q = 0.02  # 2% continuous dividend yield
div_curve = dp.DiscountCurve.flat(rate=q, end_time=2.0)

underlying_div = dp.UnderlyingData(
    initial_value=100.0,
    volatility=0.20,
    market_data=market,
    dividend_curve=div_curve,
)

bsm_div = dp.OptionValuation(
    underlying=underlying_div, spec=spec, pricing_method=dp.PricingMethod.BSM
)
print(f"BSM with q={q}: {bsm_div.present_value():.4f}")

# Compare with binomial
binom_div = dp.OptionValuation(
    underlying=underlying_div,
    spec=spec,
    pricing_method=dp.PricingMethod.BINOMIAL,
    params=dp.BinomialParams(num_steps=500),
)
print(f"Binomial with q={q}: {binom_div.present_value():.4f}")

BSM with q=0.02: 6.9869
Binomial with q=0.02: 6.9889


## 10) Discrete Cash Dividends

Discrete dividends are provided as `(ex_date, amount)` pairs. The underlying
spot is reduced by the dividend amount on each ex-date.

In [10]:
dividends = [
    (dt.datetime(2025, 9, 15), 1.50),  # $1.50 in 3 months
    (dt.datetime(2025, 12, 15), 1.50),  # $1.50 in 6 months
    (dt.datetime(2026, 3, 15), 1.50),  # $1.50 in 9 months
]

underlying_disc = dp.UnderlyingData(
    initial_value=100.0,
    volatility=0.20,
    market_data=market,
    discrete_dividends=dividends,
)

bsm_disc = dp.OptionValuation(
    underlying=underlying_disc, spec=spec, pricing_method=dp.PricingMethod.BSM
)
print(f"BSM with discrete divs: {bsm_disc.present_value():.4f}")

binom_disc = dp.OptionValuation(
    underlying=underlying_disc,
    spec=spec,
    pricing_method=dp.PricingMethod.BINOMIAL,
    params=dp.BinomialParams(num_steps=500),
)
print(f"Binomial with discrete divs: {binom_disc.present_value():.4f}")

BSM with discrete divs: 5.8361
Binomial with discrete divs: 5.8370


## 11) Puts

Switching to a put is a one-field change on `VanillaSpec`.

In [11]:
put_spec = dp.VanillaSpec(
    option_type=dp.OptionType.PUT,
    exercise_type=dp.ExerciseType.EUROPEAN,
    strike=105.0,
    maturity=maturity,
)

bsm_put = dp.OptionValuation(
    underlying=underlying, spec=put_spec, pricing_method=dp.PricingMethod.BSM
)
print(f"BSM call: {bsm.present_value():.4f}")
print(f"BSM put:  {bsm_put.present_value():.4f}")

BSM call: 8.0214
BSM put:  7.9004
